# 🧠 تدريب نموذج تصنيع الملابس - خزانتي الذكية
Smart Wardrobe AI Model Training

هذا النوت بوك يدرب نموذج MobileNetV2 لتصنيف أنواع الملابس والألوان

## المخرجات:
1. `model.tflite` — نموذج تصنيف نوع القطعة
2. `labels.txt` — تسميات الأنواع
3. `color_extractor.ts` — كود استخراج الألوان

In [ ]:
# @title 📦 تثبيت المكتبات
!pip install -q tensorflow tensorflowjs tensorflow-datasets
!pip install -q pillow numpy matplotlib opencv-python scikit-learn
print('✅ Libraries installed')

In [ ]:
# @title 🔧 استيراد المكتبات
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications
import numpy as np
import matplotlib.pyplot as plt
import os, shutil, random

print('TensorFlow:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# @title 📥 تحميل Fashion MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
print(f'Fashion MNIST: {x_train.shape[0]} تدريب, {x_test.shape[0]} اختبار')

FASHION_LABELS = [
    't-shirt', 'trouser', 'pullover', 'dress', 'coat',
    'sandal', 'shirt', 'sneaker', 'bag', 'ankle_boots'
]
print('الفئات:', FASHION_LABELS)

In [ ]:
# @title 🖼️ تجهيز البيانات لـ MobileNetV2
IMG_SIZE = 224
BATCH_SIZE = 64

# توسيع لـ 3 قنوات RGB
x_train_rgb = np.stack([x_train] * 3, axis=-1).astype('float32')
x_test_rgb = np.stack([x_test] * 3, axis=-1).astype('float32')

# تغيير الحجم إلى 224x224
x_train_resized = tf.image.resize(x_train_rgb, [IMG_SIZE, IMG_SIZE]).numpy()
x_test_resized = tf.image.resize(x_test_rgb, [IMG_SIZE, IMG_SIZE]).numpy()

# ✅ تطبيع إلى [-1, 1] كما يتوقعه MobileNetV2 مع weights='imagenet'
x_train_final = (x_train_resized - 127.5) / 127.5
x_test_final = (x_test_resized - 127.5) / 127.5

print(f'شكل بيانات التدريب: {x_train_final.shape}')
print(f'نطاق القيم: [{x_train_final.min():.2f}, {x_train_final.max():.2f}]')

In [ ]:
# @title 🏗️ بناء MobileNetV2 (Transfer Learning)
def build_model(num_classes):
    base_model = applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model, base_model

model, base_model = build_model(len(FASHION_LABELS))
model.summary()

In [ ]:
# @title 🚀 تدريب المرحلة الأولى (تجميد)
EPOCHS_FROZEN = 10

history = model.fit(
    x_train_final, y_train,
    validation_data=(x_test_final, y_test),
    epochs=EPOCHS_FROZEN,
    batch_size=BATCH_SIZE,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2),
    ]
)

# رسم منحنيات التدريب
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='تدريب')
plt.plot(history.history['val_accuracy'], label='تحقق')
plt.title('الدقة')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='تدريب')
plt.plot(history.history['val_loss'], label='تحقق')
plt.title('الخسارة')
plt.legend()
plt.show()

In [ ]:
# @title 🔥 المرحلة الثانية: Fine-tuning
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_FINETUNE = 15
history_ft = model.fit(
    x_train_final, y_train,
    validation_data=(x_test_final, y_test),
    epochs=EPOCHS_FINETUNE,
    batch_size=BATCH_SIZE // 2,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    ]
)

test_loss, test_acc = model.evaluate(x_test_final, y_test)
print(f'\n✅ الدقة النهائية على مجموعة الاختبار: {test_acc:.2%}')

In [ ]:
# @title 🔄 تحويل إلى TFLite (FP16 Quantization)
# ✅ دالة توليد البيانات الصحيحة للـ calibration
def representative_dataset_gen():
    for i in range(100):
        yield [np.expand_dims(x_train_final[i], axis=0).astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
converter.representative_dataset = representative_dataset_gen

print('🔄 جاري تحويل النموذج...')
tflite_model = converter.convert()
print(f'✅ حجم النموذج: {len(tflite_model) / 1024:.1f} KB')

with open('model_float16.tflite', 'wb') as f:
    f.write(tflite_model)

with open('labels.txt', 'w', encoding='utf-8') as f:
    for i, label in enumerate(FASHION_LABELS):
        f.write(f'{i} {label}\n')
print('✅ labels.txt saved')

In [ ]:
# @title 🧪 اختبار TFLite
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('📥 Input:', input_details[0]['shape'], input_details[0]['dtype'])
print('📤 Output:', output_details[0]['shape'], output_details[0]['dtype'])

correct = 0
total = 5
for _ in range(total):
    idx = random.randint(0, len(x_test_final) - 1)
    test_image = np.expand_dims(x_test_final[idx], axis=0).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], test_image)
    interpreter.invoke()
    prediction = interpreter.get_tensor(output_details[0]['index'])
    predicted_class = np.argmax(prediction[0])
    confidence = prediction[0][predicted_class]
    actual_class = y_test[idx]
    is_correct = predicted_class == actual_class
    if is_correct: correct += 1
    print(f'📸 الفعلي: {FASHION_LABELS[actual_class]:12s} → المتوقع: {FASHION_LABELS[predicted_class]:12s} (ثقة: {confidence:.1%}) {"✅" if is_correct else "❌"}')

print(f'\n🏆 الدقة: {correct}/{total} = {correct/total:.0%}')

In [ ]:
# @title ⬇️ تحميل النماذج النهائية
from google.colab import files
os.makedirs('model_output', exist_ok=True)
shutil.copy('model_float16.tflite', 'model_output/model.tflite')
shutil.copy('labels.txt', 'model_output/labels.txt')
!zip -j model_output/model_files.zip model_output/*

print('\n⬇️ جاري تحميل الملفات...')
files.download('model_output/model_files.zip')
print('\n📁 بعد التحميل: انسخ model.tflite + labels.txt إلى 📁 smart-wardrobe/ml/')

## 🔄 للاستخدام الإنتاجي: DeepFashion2

للحصول على دقة أعلى في تصنيف الملابس، استخدم DeepFashion2:
1. سجل في https://github.com/switchablenorms/DeepFashion2
2. حمل البيانات (13 فئة، 491K صورة)
3. ارفعها إلى Colab
4. استخدم نفس النوت بوك مع التعديلات اللازمة

In [ ]:
# @title 🌈 استخراج الألوان (K-Means)
from sklearn.cluster import KMeans

def extract_dominant_color(image, k=4):
    image = image.astype(np.uint8)
    pixels = image.reshape(-1, 3)
    kmeans = KMeans(n_clusters=k, n_init=5, random_state=42)
    kmeans.fit(pixels)
    colors = kmeans.cluster_centers_
    labels = kmeans.labels_
    counts = np.bincount(labels)
    dominant = colors[np.argmax(counts)]
    return dominant.astype(int)

def rgb_to_color_name(rgb):
    r, g, b = rgb
    if r > 200 and g > 200 and b > 200: return 'White'
    if r < 50 and g < 50 and b < 50: return 'Black'
    # باقي الألوان...
    if r > 200 and g < 100 and b < 100: return 'Red'
    if r < 100 and g > 150 and b < 100: return 'Green'
    if r < 100 and g < 100 and b > 180: return 'Blue'
    if r > 200 and g > 150 and b < 100: return 'Orange'
    if r > 150 and g < 100 and b > 150: return 'Purple'
    if r < 100 and g > 100 and b < 100: return 'Olive'
    if r > 150 and g > 100 and b < 100: return 'Brown'
    if r > 150 and g < 150 and b > 150: return 'Pink'
    if r > 200 and g > 200 and b < 150: return 'Yellow'
    if r > 150 and g > 150 and b > 150: return 'Gray'
    if r > 200 and g > 180 and b > 150: return 'Beige'
    if r < 50 and g < 80 and b > 100: return 'Navy'
    if r > 100 and g > 150 and b > 150: return 'Teal'
    if r > 200 and g < 50 and b < 50: return 'Coral'
    return 'Unknown'

sample = (x_train_final[0] * 127.5 + 127.5).astype(np.uint8)
dominant = extract_dominant_color(sample)
print(f'🎨 اللون السائد: {rgb_to_color_name(dominant)} ({dominant})')

In [ ]:
# @title 💾 حفظ color_extractor.ts
color_code = '''/**
 * استخراج اللون السائد من الصورة
 */
export function extractDominantColor(imageData: ImageData): {colorName:string;colorHex:string;confidence:number}{
  const {data,width,height}=imageData; let r=0,g=0,b=0,count=0;
  for(let y=0;y<height;y+=4)for(let x=0;x<width;x+=4){const i=(y*width+x)*4;r+=data[i];g+=data[i+1];b+=data[i+2];count++}
  return {colorName:rgbToColorName(Math.round(r/count),Math.round(g/count),Math.round(b/count)),colorHex:'#'+[Math.round(r/count),Math.round(g/count),Math.round(b/count)].map(c=>c.toString(16).padStart(2,'0')).join(''),confidence:0.85}
}
function rgbToColorName(r:number,g:number,b:number):string{
  if(r>200&&g>200&&b>200)return'White';if(r<50&&g<50&&b<50)return'Black';
  if(r>200&&g<100&&b<100)return'Red';if(r<100&&g>150&&b<100)return'Green';
  if(r<100&&g<100&&b>180)return'Blue';if(r>200&&g>150&&b<100)return'Orange';
  if(r>150&&g<100&&b>150)return'Purple';return'Unknown'}
'''
with open('/content/color_extractor.ts','w')as f:f.write(color_code)
from google.colab import files
files.download('/content/color_extractor.ts')
print('✅ color_extractor.ts downloaded')

In [ ]:
# @title 🎯 الخلاصة
print('''
═══════════════════════════════════
  ✅ تم التدريب بنجاح!
═══════════════════════════════════
📦 smart-wardrobe/ml/:
  1. model.tflite
  2. labels.txt
📦 services/ai/:
  3. color_extractor.ts
📌 بعد النقل: cd smart-wardrobe && npx expo start
''')